In [ ]:
import pandas as pd
import re as re
from quran_foundation import QuranClient

import os
import requests
from requests.auth import HTTPBasicAuth
import pyarrow

In [ ]:
API_BASE_URL = "https://apis.quran.foundation"

def get_quran_foundation_token(client_id, client_secret):
    response = requests.post(
        "https://oauth2.quran.foundation/oauth2/token",
        auth=HTTPBasicAuth(client_id, client_secret),
        headers={
            "Content-Type": "application/x-www-form-urlencoded"
        },
        data={
            "grant_type": "client_credentials",
            "scope": "content"
        },
        timeout=30
    )

    response.raise_for_status()

    return response.json()["access_token"]

CLIENT_ID = os.environ["QF_CLIENT_ID"]
CLIENT_SECRET = os.environ["QF_CLIENT_SECRET"]

access_token = get_quran_foundation_token(
    CLIENT_ID,
    CLIENT_SECRET
)

client = QuranClient(
    client_id=CLIENT_ID,
    access_token=access_token
)

In [ ]:
conversionTable = {
    # Arabic letters
    "'": "\u0621",  # ء HAMZA
    ">": "\u0623",  # أ ALEF WITH HAMZA ABOVE
    "&": "\u0624",  # ؤ WAW WITH HAMZA ABOVE
    "<": "\u0625",  # إ ALEF WITH HAMZA BELOW
    "}": "\u0626",  # ئ YEH WITH HAMZA ABOVE
    "A": "\u0627",  # ا ALEF
    "b": "\u0628",  # ب
    "p": "\u0629",  # ة TA MARBUTA
    "t": "\u062A",  # ت
    "v": "\u062B",  # ث
    "j": "\u062C",  # ج
    "H": "\u062D",  # ح
    "x": "\u062E",  # خ
    "d": "\u062F",  # د
    "*": "\u0630",  # ذ
    "r": "\u0631",  # ر
    "z": "\u0632",  # ز
    "s": "\u0633",  # س
    "$": "\u0634",  # ش
    "S": "\u0635",  # ص
    "D": "\u0636",  # ض
    "T": "\u0637",  # ط
    "Z": "\u0638",  # ظ
    "E": "\u0639",  # ع
    "g": "\u063A",  # غ
    "_": "\u0640",  # ـ TATWEEL
    "f": "\u0641",  # ف
    "q": "\u0642",  # ق
    "k": "\u0643",  # ك
    "l": "\u0644",  # ل
    "m": "\u0645",  # م
    "n": "\u0646",  # ن
    "h": "\u0647",  # ه
    "w": "\u0648",  # و
    "Y": "\u0649",  # ى ALEF MAKSURA
    "y": "\u064A",  # ي

    # Harakat / diacritics
    "F": "\u064B",  # ً FATHATAN
    "N": "\u064C",  # ٌ DAMMATAN
    "K": "\u064D",  # ٍ KASRATAN
    "a": "\u064E",  # َ FATHA
    "u": "\u064F",  # ُ DAMMA
    "i": "\u0650",  # ِ KASRA
    "~": "\u0651",  # ّ SHADDA
    "o": "\u0652",  # ْ SUKUN

    # Qur'anic / extended Buckwalter characters
    "^": "\u0653",  # ٓ MADDAH ABOVE
    "#": "\u0654",  # ٔ HAMZA ABOVE
    "`": "\u0670",  # ٰ SUPERSCRIPT ALEF
    "{": "\u0671",  # ٱ ALEF WASLA

    ":": "\u06DC",  # ۜ SMALL HIGH SEEN
    "@": "\u06DF",  # ۟ SMALL HIGH ROUNDED ZERO
    '"': "\u06E0",  # ۠ SMALL HIGH UPRIGHT RECTANGULAR ZERO
    "[": "\u06E2",  # ۢ SMALL HIGH MEEM ISOLATED FORM
    ";": "\u06E3",  # ۣ SMALL LOW SEEN
    ",": "\u06E5",  # ۥ SMALL WAW
    ".": "\u06E6",  # ۦ SMALL YEH
    "!": "\u06E8",  # ۨ SMALL HIGH NOON
    "-": "\u06EA",  # ۪ EMPTY CENTRE LOW STOP
    "+": "\u06EB",  # ۫ EMPTY CENTRE HIGH STOP
    "%": "\u06EC",  # ۬ ROUNDED HIGH STOP WITH FILLED CENTRE
    "]": "\u06ED",  # ۭ SMALL LOW MEEM

    # Standard Buckwalter ALEF WITH MADDA.
    # QAC notes this isn't used in the Tanzil XML/JQuranTree encoding,
    # but retaining it makes the converter more complete.
    "|": "\u0622",  # آ

    " ": " ",
}

surahs = '''1 - Al-Fatiha (The opener)
2 - Al-Baqarah (The cow)
3 - Al-Imran (Family of Imran)
4 - An-Nisa (The Women)
5 - Al-Ma'idah (The Table Spread)
6 - Al-Anam (The Cattle)
7 - Al-A'raf (The Heights)
8 - Al-Anfal (The Spoils of War)
9 - At-Taubah (The Repentance)
10 - Yunus (Jonah)
11 - Hud (Hud)
12 - Yusuf (Joseph)
13 - Ar-Ra'd (Thunder)
14 - Ibrahim (Abraham)
15 - Al-Hijr (The Stoneland)
16 - An-Nahl (The Bee)
17 - Al-Isra (The Night Journey)
18 - Al-Kahf (The Cave)
19 - Maryam (Mary)
20 - Ta-Ha (Ta-Ha)
21 - Al-Anbiya (The Prophets)
22 - Al-Hajj (The Pilgrimage)
23 - Al-Mu'minun (The Believers)
24 - An-Nur (The Light)
25 - Al-Furqan (The Criterion)
26 - Ash-Shu'ara (The Poets)
27 - An-Naml (The Ants)
28 - Al-Qasas (The Story)
29 - Al-Ankabut (Spider)
30 - Ar-Rum (The Romans)
31 - Luqman (Luqman)
32 - As-Sajdah (Prostration)
33 - Al-Ahzab (The Confederates)
34 - Saba (Sheba)
35 - Fatir (The Originator)
36 - Ya-Sin (Ya Sin)
37 - As-Saffat (Those Who Set the Ranks)
38 - Sad (The letter Saad)
39 - Az-Zumar (The Troops)
40 - Ghafir (The Forgiver)
41 - Fussilat (Explained in Detail)
42 - Ash-Shura (The Consultation)
43 - Az-Zukhruf (The Ornaments of Gold)
44 - Ad-Dukhan (The Smoke)
45 - Al-Jathiyah (The Crouching)
46 - Al-Ahqaf (The Wind Curved Sandhill)
47 - Muhammad (Muhammad)
48 - Al-Fath (The Victory)
49 - Al-Hujurat (The Private Chambers)
50 - Qaf (Qaf)
51 - Adh-Dhariyat (The Scatterers)
52 - At-Tur (The Mountain)
53 - An-Najm (The Star)
54 - Al-Qamar (The Moon)
55 - Ar-Rahman (The Beneficent)
56 - Al-Waqi'ah (The Inevitable)
57 - Al-Hadid (The Iron)
58 - Al-Mujadila (The Pleading Women)
59 - Al-Hashr (The Exile)
60 - Al-Mumtahanah (She That is to be Examined)
61 - As-Saff (The Ranks)
62 - Al-Jumu'ah (Congregation Prayer)
63 - Al-Munafiqun (The Hypocrites)
64 - At-Taghabun (Mutual Disposession)
65 - At-Talaq (The Divorce)
66 - At-Tahrim (The Prohibition)
67 - Al-Mulk (The Sovereignty)
68 - Al-Qalam (The Pen)
69 - Al-Haqqah (The Reality)
70 - Al-Ma'arij (The Ascending Stairways)
71 - Nuh (Noah)
72 - Al-Jinn (The Jinn)
73 - Al-Muzzammil (The Enshrouded One)
74 - Al-Muddaththir (The Cloaked One)
75 - Al-Qiyamah (The Resurrection)
76 - Al-Insan (The Man)
77 - Al-Mursalat (The Emissaries)
78 - An-Naba (The Tidings)
79 - An-Nazi'at (Those who drag forth)
80 - Abasa (He Frowned)
81 - At-Takwir (The Overthrowing)
82 - Al-Infitar (The Cleaving)
83 - Al-Mutaffifin (The Defrauding)
84 - Al-Inshiqaq (The Sundering)
85 - Al-Buruj (The Mansions of the Stars)
86 - At-Tariq (The Nightcommer)
87 - Al-Ala (The Most High)
88 - Al-Ghashiyah (The Overwhelming)
89 - Al-Fajr (The Dawn)
90 - Al-Balad (The City)
91 - Ash-Shams (The Sun)
92 - Al-Lail (The Night)
93 - Ad-Duha (The Morning Brightness)
94 - Ash-Sharh (The Expansion)
95 - At-Tin (The Fig)
96 - Al-Alaq (The Blood Clot)
97 - Al-Qadr (The Power)
98 - Al-Bayyina (The Evidence)
99 - Az-Zalzalah (The Earthquake)
100 - Al-Adiyat (The Courser)
101 - Al-Qari'ah (The Calamity)
102 - At-Takathur (Vying for increase)
103 - Al-Asr (The Declining Day)
104 - Al-Humazah (The Slanderer)
105 - Al-Fil (The Elephant)
106 - Quraysh (Quraish)
107 - Al-Ma'un (The Small Kindness)
108 - Al-Kawthar (The Abundance)
109 - Al-Kafirun (The Disbelievers)
110 - An-Nasr (The Divine Support)
111 - Al-Masad (The Palm Fiber)
112 - Al-Ikhlas (The Sincerity)
113 - Al-Falaq (The Daybreak)
114 - An-Nas (The Mankind)'''

surahs = [x.split(" - ")[1].split(" (")[0] for x in surahs.split("\n")]

def convert_buckwalter_to_arabic(s):
    return "".join(conversionTable[x] for x in s)

In [ ]:
database = pd.read_csv(
    "./quranic-corpus-morphology-0.4.txt",
    skiprows=57,
    sep="\t",
    header=None,
    keep_default_na=False
)

database.columns = ["Position", "ArabicBuckwalter", "POS", "Morphology"]

database["Arabic"] = database["ArabicBuckwalter"].apply(
    convert_buckwalter_to_arabic
)

database

In [ ]:
def ParseMorphology(text):
    tags = text.split("|")
    toReturn = {}

    for tag in tags:
        if tag in ["STEM", "PREFIX", "SUFFIX"]:
            toReturn["Entry Type"] = tag
        elif tag in ["Al+", "bi+", "ka+", "ta+", "sa+", "ya+", "ha+", "A:INTG+", "A:EQ+", "w:CONJ+", "w:REM+", "w:CIRC+", \
                "w:SUP+", "w:P+", "w:COM+", "f:REM+", "f:CONJ+", "f:RSLT+", "f:SUP+", "f:CAUS+", "l:P+", "l:EMPH+", "l:PRP+", "l:IMPV+"]:
            toReturn["Prefix Type"] = tag
        elif tag.startswith("PRON:"):
            toReturn["Suffix Type"] = tag[5:]
        elif tag.startswith("POS:"):
            toReturn["Part of Speech"] = tag[4:]
        elif tag.startswith("LEM:"):
            lemma_id = tag[4:]

            match = re.fullmatch(r"(.+?)(\d+)?", lemma_id)

            lemma_buckwalter = match.group(1)
            lemma_discriminator = match.group(2)

            toReturn["Lemma ID"] = lemma_id
            toReturn["Lemma Buckwalter"] = lemma_buckwalter
            toReturn["Lemma"] = convert_buckwalter_to_arabic(lemma_buckwalter)

            if lemma_discriminator is not None:
                toReturn["Lemma Discriminator"] = int(lemma_discriminator)
        elif tag.startswith("ROOT:"):
            toReturn["Root"] = convert_buckwalter_to_arabic(tag[5:])
        elif tag.startswith("SP:"):
            toReturn["Special Group"] = convert_buckwalter_to_arabic(tag[3:])
        elif match := re.fullmatch(r"([123])?([MF])?([SDP])?", tag):
            person, gender, number = match.groups()

            if person:
                toReturn["Person"] = person

            if gender:
                toReturn["Gender"] = gender

            if number:
                toReturn["Number"] = number
        elif tag in ["NOM", "ACC", "GEN"]:
            toReturn["Case"] = tag
        elif tag in ["ACT", "PASS"]:
            toReturn["Voice"] = tag
        elif tag in ["PCPL", "VN"]:
            toReturn["Derived Noun Type"] = tag
        elif tag in ["PERF", "IMPF", "IMPV"]:
            toReturn["Verb Tense"] = tag
        elif re.fullmatch("\([IVX]+\)", tag):
            toReturn["Derived Verb Form"] = tag
        elif tag in ["DEF", "INDEF"]:
            toReturn["Definiteness"] = tag
        elif tag.startswith("MOOD:"):
            toReturn["Verb Mood"] = tag[5:]
        elif tag == "+n:EMPH":
            toReturn["Noon of Emphasis"] = "Y"
        elif tag == "+VOC":
            toReturn["Vocative Suffix"] = "Y"
        else:
            raise ValueError(f"{tag} not found.")

    return toReturn


In [ ]:
database["Morphology Info"] = database["Morphology"].apply(ParseMorphology)

database[["Surah", "Ayah", "Word", "Segment"]] = (
    database["Position"]
    .str.strip("()")
    .str.split(":", expand=True)
    .astype(int)
)

database["SurahName"] = database["Surah"].apply(
    lambda x: surahs[x - 1]
)

morphology_columns = pd.json_normalize(
    database["Morphology Info"]
).add_prefix("Morph_")

database_parquet = pd.concat(
    [
        database.drop(columns=["Morphology Info"]),
        morphology_columns
    ],
    axis=1
)

database_parquet.to_parquet(
    "qac_morphology.parquet",
    index=False
)

database

In [ ]:
pronoun_pos = {"PRON", "DEM", "REL"}

def is_vocabulary_entry(info):
    # Normal lemma-bearing vocabulary
    if "Lemma" in info:
        return True

    # Independent pronouns, but NOT attached pronoun suffixes
    if (
        info.get("Entry Type") == "STEM"
        and info.get("Part of Speech") in pronoun_pos
    ):
        return True

    return False

def get_vocabulary_id(row):
    info = row["Morphology Info"]

    if "Lemma ID" in info:
        return info["Lemma ID"]

    if (
        info.get("Entry Type") == "STEM"
        and info.get("Part of Speech") in {"PRON", "DEM", "REL"}
    ):
        return f'{info["Part of Speech"]}:{row["ArabicBuckwalter"]}'

    return None


def get_headword(row):
    info = row["Morphology Info"]

    if "Lemma" in info:
        return info["Lemma"]

    if (
        info.get("Entry Type") == "STEM"
        and info.get("Part of Speech") in {"PRON", "DEM", "REL"}
    ):
        return row["Arabic"]

    return None


vocabulary_occurrences = database[database["Morphology Info"].apply(is_vocabulary_entry)].copy()

vocabulary_occurrences["Vocabulary ID"] = (
    vocabulary_occurrences.apply(get_vocabulary_id, axis=1)
)

vocabulary_occurrences["Lemma"] = (
    vocabulary_occurrences.apply(get_headword, axis=1)
)

vocabulary_occurrences

In [ ]:
def get_surah_words(surah: int):
    qf_words = []
    page = 1

    while True:
        response = requests.get(
            f"{API_BASE_URL}/content/api/v4/verses/by_chapter/{surah}",
            headers={
                "x-auth-token": access_token,
                "x-client-id": CLIENT_ID
            },
            params={
                "words": "true",
                "per_page": 50,
                "page": page
            },
            timeout=30
        )

        if not response.ok:
            print("URL:", response.url)
            print("Status:", response.status_code)
            print("Response:", response.text)
            response.raise_for_status()

        result = response.json()

        for verse in result["verses"]:
            for word in verse["words"]:
                if word["char_type_name"] != "word":
                    continue

                qf_words.append({
                    "Surah": surah,
                    "Ayah": verse["verse_number"],
                    "Word": word["position"],
                    "English": word["translation"]["text"],
                    "Audio": word["audio_url"],
                    "Juz": verse["juz_number"],
                    "Hizb": verse["hizb_number"],
                    "Rub el Hizb": verse["rub_el_hizb_number"],
                    "Ruku": verse["ruku_number"]
                })

        next_page = result["pagination"]["next_page"]

        if next_page is None:
            break

        page = next_page

    return qf_words

In [ ]:
all_qf_words = []

for surah in range(1, 115):
    print(f"Downloading surah {surah}...")
    all_qf_words.extend(get_surah_words(surah))

qf_database = pd.DataFrame(all_qf_words)

In [ ]:
key = ["Surah", "Ayah", "Word"]

# Quran Foundation should have one record per written Quranic word
assert not qf_database.duplicated(key).any()

# Make sure every requested surah arrived
assert set(qf_database["Surah"]) == set(range(1, 115))

print(f"Quran Foundation words: {len(qf_database):,}")
print(f"QAC lemma segments: {len(vocabulary_occurrences):,}")

merged = pd.merge(
    vocabulary_occurrences,
    qf_database,
    on=key,
    how="left",
    validate="many_to_one",
    indicator=True
)

print(merged["_merge"].value_counts())

In [ ]:
key = ["Surah", "Ayah", "Word"]

# Every vocabulary occurrence successfully matched Quran Foundation
assert (merged["_merge"] == "both").all()

# Quran Foundation has no duplicate word identifiers
assert not qf_database.duplicated(key).any()

# Find Quran Foundation words with no vocabulary-bearing QAC entry
qf_unmatched = (
    qf_database
    .merge(
        vocabulary_occurrences[key].drop_duplicates(),
        on=key,
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
)

# All intentionally excluded words should be Quranic initials
assert len(qf_unmatched) == 30

lemma_keys = (
    vocabulary_occurrences[key]
    .drop_duplicates()
)

qf_without_lemma = (
    qf_database
    .merge(
        lemma_keys,
        on=key,
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

no_lemma_analysis = qf_without_lemma.merge(
    database,
    on=key,
    how="left"
)

assert set(no_lemma_analysis["POS"]) == {"INL"}

In [ ]:
merged["Root"] = (
    merged["Morphology Info"]
    .apply(lambda x: x.get("Root"))
)

merged["Number"] = (
    merged["Morphology Info"]
    .apply(lambda x: x.get("Number"))
)

merged["Gender"] = (
    merged["Morphology Info"]
    .apply(lambda x: x.get("Gender"))
)

merged["Person"] = (
    merged["Morphology Info"]
    .apply(lambda x: x.get("Person"))
)

In [ ]:
merged.to_csv("./QuranTotalMergedDatabase.csv", index=None)
merged

In [ ]:
qf_database.to_csv("./QF_Database.csv", index=None)